# 01 - Data Cleaning
**QSS 20 Final Project | Olivia Tak**

This notebook filters the dataset to 2023 trending videos, deduplicates, 
log transforms virality metrics, and merges category labels.

## Imports

In [1]:
import pandas as pd
import numpy as np
import json

## Load raw data

In [2]:
# load raw CSV
df = pd.read_csv('../data/US_youtube_trending_data.csv')
df['trending_date'] = pd.to_datetime(df['trending_date'])
print(df.shape)

(268787, 16)


## Filter to 2023

In [3]:
# filter to videos that trended in 2023
df_2023 = df[df['trending_date'].dt.year == 2023].copy()
print(f"After filtering to 2023: {len(df_2023)} rows")

After filtering to 2023: 72397 rows


## Count days trending and deduplicate

In [4]:
# count how many days each video appeared on trending
days_trending = df_2023.groupby('video_id')['trending_date'].count().reset_index()
days_trending.columns = ['video_id', 'days_trending']
print(days_trending['days_trending'].describe())

count    12176.000000
mean         5.945877
std          1.869705
min          1.000000
25%          5.000000
50%          6.000000
75%          7.000000
max         37.000000
Name: days_trending, dtype: float64


In [5]:
# deduplicate keeping first trending appearance, then merge count back in
df_2023 = df_2023.sort_values('trending_date')
df_2023 = df_2023.drop_duplicates(subset='video_id', keep='first')
df_2023 = df_2023.merge(days_trending, on='video_id', how='left')

print(f"After deduplication: {len(df_2023)} rows")
print(df_2023[['title', 'days_trending']].head(10))

After deduplication: 12176 rows
                                               title  days_trending
0  Peach Bowl: Ohio State Buckeyes vs. Georgia Bu...              9
1                              Tom MacDonald - Ghost              4
2  Ñengo Flow, Bad Bunny - Gato de Noche ( Video ...              4
3  Seattle weather conditions: Ongoing power outa...              4
4         Honda’s $225 Million Mistake – Rune Review              4
5                         i became an italian farmer              4
6             What Dan and Phil Text Each Other 2022              4
7  It’s Beginning To Look A Lot Like Christmas (c...              4
8        That '90s Show | Official Trailer | Netflix              4
9          20 Things You Didn't Know About The NBA..              4


## Log transform virality metrics

In [8]:
# log transformation of view_count to account for right skew
df_2023['log_views'] = np.log(df_2023['view_count'] + 1)

# same for likes and comment count
df_2023['log_likes'] = np.log(df_2023['likes'] + 1)
df_2023['log_comments'] = np.log(df_2023['comment_count'] + 1)

print(df_2023[['likes', 'log_likes', 'comment_count', 'log_comments']].describe().round(2))

            likes  log_likes  comment_count  log_comments
count    12176.00   12176.00       12176.00      12176.00
mean     67326.57      10.10        5054.06          7.50
std     220854.46       1.46       21592.82          1.46
min          0.00       0.00           0.00          0.00
25%      11966.75       9.39         937.00          6.84
50%      23988.50      10.09        1840.00          7.52
75%      52986.00      10.88        3777.50          8.24
max    7114337.00      15.78     1086344.00         13.90


## Load category labels

In [9]:
# load category JSON and map category names onto the dataframe
with open('../data/US_category_id.json', 'r') as f:
    category_data = json.load(f)

category_dict = {int(item['id']): item['snippet']['title'] 
                 for item in category_data['items']}

df_2023['category'] = df_2023['categoryId'].map(category_dict)

print(df_2023['category'].value_counts())

category
Gaming                   2550
Entertainment            2343
Music                    1850
Sports                   1773
People & Blogs            947
Film & Animation          500
Comedy                    482
News & Politics           362
Science & Technology      355
Autos & Vehicles          303
Education                 299
Howto & Style             259
Travel & Events            98
Pets & Animals             54
Nonprofits & Activism       1
Name: count, dtype: int64


## Save cleaned data


In [10]:
# save cleaned dataframe to data folder for use in next notebooks
df_2023.to_csv('../data/df_2023_clean.csv', index=False)
print("Saved successfully")
print(f"Final shape: {df_2023.shape}")

Saved successfully
Final shape: (12176, 21)
